# 1 · Data — the response table and the expression matrix

The first of five stages. Everything downstream is derived from what this notebook writes, and
nothing in it depends on a model, a panel or a representation.

**What this stage does.**

- **`fetch`** — download CTRPv2's reprocessed dose-response data from a **pinned** Zenodo record and
  verify it against that record's published MD5s. Idempotent, and the only step that touches the
  network.
- **`convert`** — turn SCP542's distributed text matrices into `SCP542_CCLE.h5ad`, reduced to this
  variant's highly variable genes.

**What it deliberately does not do.** No scGPT embedding, no CTRP targets, no splits, no PCA —
those are [stage 3](3_representations.ipynb). The split is not cosmetic:
[stage 2](2_drug_selection.ipynb) needs exactly two things, the cell-line roster this notebook
writes and the response table it fetches, so the drug panel can be built **before** any
representation exists and without ever consulting a model.

**The steps are functions, not cells.** Each is one call into `scripts/preprocessing/pipeline.py`,
where every step owns its own guard and preconditions — so running these out of order fails loudly
instead of producing something subtly wrong. `run_preprocessing.py`, the CLI that used to hold the
step order, was [archived](../scripts/archive/README.md) on 12.08.2026: the order is now the
numbering of these notebooks, and keeping a second copy of it in a CLI meant keeping two things in
step.

> ⛔ **Nothing here has been re-run.** The 03.08.2026 freeze in [TODO](../docs/TODO.md) holds until
> Selin's review finishes; every artifact on disk predates the code that now writes it. This
> notebook is the definition of the stage, not a record of a run.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.layout import DEFAULT_CTRP_SCORE, DEFAULT_VARIANT, PipelinePaths
from scripts.preprocessing import pipeline

# The one variant the pipeline runs on. The hvg1000/2000/3000 sweep is built in
# analysis/qc/, next to the only notebook that consumes it -- a stage everyone runs
# should not carry an off-by-default branch for one analysis (Selin, 12.08.2026).
VARIANT = DEFAULT_VARIANT

# Carried so the paths object is complete, but NOT used by this stage: fetch and convert
# are both target-agnostic. The score first bites in stage 3, which writes one targets
# h5ad per measure.
SCORE = DEFAULT_CTRP_SCORE


# ⚠️ SET TO True FOR R2 ON 13.08.2026, AND IT MUST GO BACK TO False AFTERWARDS.
#
# `convert` and `scgpt` refuse to replace an existing artifact, because everything downstream
# derives from them: a silent rebuild invalidates every representation and target while leaving them
# on disk looking current. That guard is the reason this constant exists, and True disarms it for
# every subsequent run of this notebook, not only the intended one.
#
# It is True because R2 IS the genuine rebuild -- every artifact under data/processed/ predates the
# code that now produces it. Returning it to False is part of closing R2; it is written here rather
# than only in TODO.md because this line is what a later reader will actually run.
OVERWRITE = True

# THE VARIANTS R1 DECIDED (Selin, 12.08.2026): hvg5000 AND all_genes -- not all five, not
# hvg5000 alone. hvg5000 is the default training variant and all_genes is what the report's
# full-transcriptome numbers rest on; hvg1000/2000/3000 keep their current artifacts and are
# re-embedded later as a top-up if the gene-set sweep is to be like-for-like (docs/TODO.md, R1).
#
# Parameterised 13.08.2026. Until then both drivers ran the single `DEFAULT_VARIANT`, so covering
# R1's decision meant running the notebook twice and remembering to edit a constant in between --
# which is how a variant silently goes missing from a rerun. The loop is the record of the decision.
VARIANTS_TO_RUN = ('hvg5000', 'all_genes')
PATHS = {v: PipelinePaths.build(None, v, SCORE) for v in VARIANTS_TO_RUN}

# `fetch` is variant-agnostic -- it writes into data/metadata/, shared by every variant -- so it
# takes one paths object rather than the loop.
paths = PATHS[VARIANT]
print(f'data root : {paths.data_root}')
print(f'convert   : {len(VARIANTS_TO_RUN)} variants -> {list(VARIANTS_TO_RUN)}')
print(f'variant   : {paths.variant} -> {paths.processed_dir}')
print(f'score     : {paths.score}  (unused here; stage 3 writes {paths.targets_h5ad.name})')

## A · The response table

CTRPv2's dose-response data, as **reprocessed by DrEval**: they take CTRPv2's raw measurements,
normalise each replicate against its own no-drug control, and re-fit every curve with CurveCurator —
folding replicate disagreement into the fit rather than averaging replicates before fitting, which
their Methods argue "leads to inaccurate or misleading drug response measures in the case of large
discrepancies between replicates".

**The record is pinned, and that is a decision rather than a default.** `drevalpy`'s own loader
resolves the concept DOI to whatever is latest, so the data it returns changes underneath you.
`layout.ZENODO_RESPONSE_RECORD` fixes one record; the cache directory carries its number, so an older
copy is visible on disk rather than silently overwritten. Bumping it is a **target change** — every
number downstream has to be re-derived and [Step 01](../docs/steps/01-datasets-and-harmonization.md)
has to record the new record and retrieval date.

Both archives are verified against the record's published MD5s and refuse to proceed on a mismatch,
so this is safe to re-run: an archive already cached and matching is neither re-downloaded nor
re-extracted.

In [ ]:
response_csv = pipeline.fetch(paths)
response_csv

In [ ]:
import json

# What was actually retrieved, read from the data rather than from the code that fetched it --
# fetch() writes this alongside the archives so the version is legible from the artifact.
prov = json.loads((paths.drevalpy_dir / 'provenance.json').read_text())
for k in ('zenodo_record', 'doi', 'publication_date', 'retrieved'):
    print(f'{k:18s} {prov[k]}')
print(f'{"files":18s} {", ".join(prov["files"])}')

## B · The expression matrix

SCP542 is distributed as **counts per million**, so library-size normalisation has already been
applied — once, to the full gene matrix, which is where it belongs. This step does not normalise a
second time.

Gene selection runs on the SCP542 authors' own quantification, $E_{ij} = \log_2(1 + \mathrm{CPM}_{ij}/10)$.
The divisor is theirs and is argued from a measured property of the data: the average number of
unique molecular identifiers per cell is below $10^5$, and without it the difference between a
detected and an undetected gene is inflated. The top **`VARIANT`** genes by the dispersion-based
Seurat criterion are kept.

**The transform is applied to a copy.** The ranking happens on the transformed matrix; the saved
`.X` keeps the original CPM values, because scGPT in stage 3 reads only the *order* of a cell's
values and any strictly increasing map leaves that order — and therefore its input — unchanged.

⚠️ **`convert` refuses to overwrite.** If `SCP542_CCLE.h5ad` already exists, the next cell raises
`FileExistsError` rather than rebuilding it. That is deliberate: every representation and target
downstream derives from this file, so replacing it silently invalidates them while leaving them on
disk looking current. Pass `overwrite=True` only if you mean it.

> ⚠️ **`OVERWRITE` is currently `True`, and that guard is therefore disarmed (13.08.2026).** It was
> set for R2, which is a genuine rebuild — every artifact under `data/processed/` predates the code
> that now produces it. **Setting it back to `False` is part of closing R2.** Left `True`, the next
> person to run this notebook rebuilds the foundation of every downstream artifact without being
> asked to confirm.

**It runs once per variant.** `VARIANTS_TO_RUN` is `('hvg5000', 'all_genes')` — R1's decision
(Selin, 12.08.2026) — and the loop is what makes that decision executable. Until 13.08.2026 both
drivers ran the single `DEFAULT_VARIANT`, so covering R1 meant running the notebook twice and
remembering to edit a constant in between, which is how a variant silently goes missing from a
rerun. `fetch` stays outside the loop: it writes into `data/metadata/`, shared by every variant.

In [ ]:
# One call per variant. `convert` guards each output itself, so a variant already built raises
# rather than being rebuilt silently -- see OVERWRITE above.
raw_h5ad = {v: pipeline.convert(PATHS[v], overwrite=OVERWRITE) for v in VARIANTS_TO_RUN}
raw_h5ad

In [ ]:
import anndata as ad

# The distributed gene count used to be written here as a literal ("of SCP542's 22,722
# distributed"). Removed 13.08.2026: convert() already prints `Gene count: <before> -> <after>`
# from the matrix it just read, one cell above, so the literal was a second copy of a number the
# run computes -- and the copy is the one that goes stale when the source or the HVG step moves.
#
# obs columns are printed too (13.08.2026): total_counts and pct_counts_mt are written here by
# convert, from the raw UMI matrix, and their absence is what makes 4b's confound veto unevaluable
# -- so this is the cell where a missing one should be noticed, not four stages later.
for v in VARIANTS_TO_RUN:
    a = ad.read_h5ad(raw_h5ad[v], backed='r')      # backed: .X stays on disk
    lines = a.obs['Cell_line'].astype(str).str.split('_').str[0].nunique()
    qc = [c for c in ('total_counts', 'pct_counts_mt') if c in a.obs.columns]
    print(f'--- {v} ---')
    print(f'cells      : {a.n_obs:,}')
    print(f'genes kept : {a.n_vars:,}')
    print(f'cell lines : {lines}')
    print(f'.X         : {a.X.dtype}, still CPM -- max {a.X[:200].max():.1f}')
    print(f'var cols   : {list(a.var.columns)}')
    print(f'UMI qc     : {qc if qc else "MISSING -- 4b stage 6 cannot run"}')
    a.file.close()